In [13]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from cdt.data import AcyclicGraphGenerator
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.GraphUtils import GraphUtils
from causallearn.graph.GeneralGraph import GeneralGraph
from collections import Counter
from pathlib import Path
import os

In [8]:
project_root = Path('./').resolve().parents[2]
base_dir = os.path.join(project_root, 'mastersResearch_Jun_Sept2025', 'causalDiscovery_J')
print(base_dir)

/Users/joshparchure/pyspocClone5Aug/py-spoc/working_analyses/mastersResearch_Jun_Sept2025/causalDiscovery_J


In [9]:
chain_folder = os.path.join(base_dir, 'dataGen', 'chain100data')
loop_folder = os.path.join(base_dir, 'dataGen', 'loop100data')

In [ ]:
def run_pc_on_folder(folder_path, max_files=100):
    """
    Run the PC algorithm on datasets in a folder.

    Iterates through up to `max_files` CSV files in the specified folder, runs the PC algorithm
    on each dataset, and collects the resulting adjacency matrices and graph objects.

    Args:
        folder_path (str): Path to the folder containing dataset CSV files.
        max_files (int): Maximum number of files to process.

    Returns:
        tuple: (adjacency_matrices, graph_objects)
            adjacency_matrices (list): List of adjacency matrices for each dataset.
            graph_objects (list): List of causal-learn Graph objects for each dataset.
    """
    adjacency_matrices = []
    graph_objects = []
    files = sorted(os.listdir(folder_path))[:max_files]  # limit to max_files

    for i, filename in enumerate(files):
        filepath = os.path.join(folder_path, filename)
        df = pd.read_csv(filepath)
        data = df.to_numpy()
        
        cg = pc(data)
        adjacency_matrices.append(cg.G.graph.copy())  # for later comparison
        graph_objects.append(cg.G)                    # for drawing
        
        if (i + 1) % 5 == 0:
            print(f"  Processed {i + 1}/{len(files)} files")
    
    return adjacency_matrices, graph_objects

def group_graphs_by_structure(graphs):
    """
    Group causal-learn Graph objects by their adjacency matrix structure.

    Args:
        graphs (list): List of causal-learn Graph objects.

    Returns:
        dict: Dictionary mapping unique adjacency matrix tuples to lists of Graph objects.
    """
    grouped = {}
    for G in graphs:
        adj = tuple(map(tuple, G.graph))  # hashable matrix
        if adj not in grouped:
            grouped[adj] = []
        grouped[adj].append(G)
    return grouped

def print_dag_distribution(dag_groups, label=''):
    """
    Print the distribution of unique DAG structures in grouped results.

    Args:
        dag_groups (dict): Dictionary mapping adjacency matrix tuples to lists of Graph objects.
        label (str): Optional label for the output.
    """

    print(f"\nDAG distribution for {label}:")
    for i, (adj, graphs) in enumerate(dag_groups.items(), 1):
        print(f"  DAG {i}: {len(graphs)} occurrences")

def save_example_dags_from_groups(dag_groups, folder_name='dag_images', prefix='dag', max_examples=3):
    """
    Save example DAG visualizations from each group of unique structures.

    Args:
        dag_groups (dict): Dictionary mapping adjacency matrix tuples to lists of Graph objects.
        folder_name (str): Directory to save the images.
        prefix (str): Prefix for saved image filenames.
        max_examples (int): Maximum number of unique DAGs to save.
    """

    os.makedirs(folder_name, exist_ok=True)

    for i, (adj, graph_list) in enumerate(dag_groups.items()):
        if i >= max_examples:
            break
        G = graph_list[0]  # Take any one of them to draw
        pyd = GraphUtils.to_pydot(G)
        filename = os.path.join(folder_name, f"{prefix}_{i+1}_count{len(graph_list)}.png")
        pyd.write_png(filename)
        print(f"Saved {filename}")

In [ ]:
print("Running PC on chain data...")
chain_adjs, chain_graphs = run_pc_on_folder(chain_folder, max_files=10)
print("Running PC on loop data...")
loop_adjs, loop_graphs = run_pc_on_folder(loop_folder, max_files=10)

# Group DAGs by unique structure
chain_groups = group_graphs_by_structure(chain_graphs)
loop_groups = group_graphs_by_structure(loop_graphs)

# Print distribution summaries
print_dag_distribution(chain_groups, label='Chain')
print_dag_distribution(loop_groups, label='Loop')

In [ ]:
# Save example DAG visualizations
save_example_dags_from_groups(chain_groups, folder_name='chain_dags', prefix='chain')
save_example_dags_from_groups(loop_groups, folder_name='loop_dags', prefix='loop')

Saved chain_dags/chain_1_count1.png
Saved chain_dags/chain_2_count1.png
Saved chain_dags/chain_3_count1.png
Saved loop_dags/loop_1_count1.png
Saved loop_dags/loop_2_count1.png
Saved loop_dags/loop_3_count1.png
